# 06 — Production Pipeline, UI and F1 25 Migration

A model becomes a system only when its data, artifacts, execution states, interfaces and failures are controlled.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
import sys
sys.path.insert(0, str(ROOT / 'src'))

In [2]:
summary = json.loads((ROOT/'artifacts/runs/reference_gru/summary.json').read_text())
list((ROOT/'artifacts/runs/reference_gru').iterdir())

[PosixPath('/mnt/data/Project_APEX_F1_Simulation_Engine/artifacts/runs/reference_gru/quality_report.json'),
 PosixPath('/mnt/data/Project_APEX_F1_Simulation_Engine/artifacts/runs/reference_gru/model'),
 PosixPath('/mnt/data/Project_APEX_F1_Simulation_Engine/artifacts/runs/reference_gru/publication.json'),
 PosixPath('/mnt/data/Project_APEX_F1_Simulation_Engine/artifacts/runs/reference_gru/metrics.json'),
 PosixPath('/mnt/data/Project_APEX_F1_Simulation_Engine/artifacts/runs/reference_gru/ablations.json'),
 PosixPath('/mnt/data/Project_APEX_F1_Simulation_Engine/artifacts/runs/reference_gru/summary.json'),
 PosixPath('/mnt/data/Project_APEX_F1_Simulation_Engine/artifacts/runs/reference_gru/splits.json'),
 PosixPath('/mnt/data/Project_APEX_F1_Simulation_Engine/artifacts/runs/reference_gru/rollout_preview.csv'),
 PosixPath('/mnt/data/Project_APEX_F1_Simulation_Engine/artifacts/runs/reference_gru/canonical_telemetry.csv'),
 PosixPath('/mnt/data/Project_APEX_F1_Simulation_Engine/artifacts/ru

## Idempotent stage flow

```text
ingest -> quality -> split/scale -> train -> evaluate -> ablate -> publish
```

A stage checks for its immutable output before recomputing. Retries therefore do not silently duplicate or corrupt work. The CLI, tests and Airflow wrapper call the same stage functions.

In [3]:
from apexsim.registry import RunRegistry
registry = RunRegistry(ROOT/'artifacts/runs/runs.sqlite')
pd.DataFrame(registry.list_runs())[['run_id','status','model_kind','started_at']]

,run_id,status,model_kind,started_at
0,reference_rssm,succeeded,rssm,2026-08-01T20:14:48.366415+00:00
1,reference_ssm,succeeded,ssm,2026-08-01T20:14:03.381417+00:00
2,reference_gru,succeeded,gru,2026-08-01T20:13:18.084900+00:00


## UI boundary

The UI is a client of stable artifacts and simulation functions. It does not contain training logic. This lets you later replace Gradio with React/Next.js without rewriting ingestion or modelling.

## F1 25 migration map

When the game becomes available:

1. Add a UDP listener and packet decoder.
2. Preserve packet/session/frame identifiers and monotonic timestamps.
3. Translate motion, telemetry, status, setup, damage and tyre packets into an expanded canonical contract.
4. Replace steering proxy with actual steering input.
5. Add wheel-level temperatures, pressures, slip, suspension and damage.
6. Train on player-specific trajectories.
7. Add a reward/cost head for lap time, track limits, tyre life and energy.
8. Add actor/critic or MPC only after the learned dynamics pass closed-loop validation.

In [4]:
print('Launch the interface with:')
print('apexsim ui --run-dir artifacts/runs/reference_gru --config configs/fast.yaml')
print('Launch the API with:')
print('apexsim api --artifacts-dir artifacts/runs')

Launch the interface with:
apexsim ui --run-dir artifacts/runs/reference_gru --config configs/fast.yaml
Launch the API with:
apexsim api --artifacts-dir artifacts/runs


### Final independence challenge
Create a new adapter for a CSV with different column names and units. Do not edit the model. If the new adapter reaches the canonical contract and the rest of the system still runs, the architecture has achieved source independence.